In [ ]:
# %% [markdown]
# # GPU-Optimized Multimodal RAG with ColPali + MedGemma - Leishmania Focus

# %%
# 1) Install dependencies and authentication
# -----------------------------------------------------------------
!sudo apt-get update && sudo apt-get install -y poppler-utils dialog

# Authenticate with Hugging Face
from huggingface_hub import login
from getpass import getpass

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HUGGINGFACE_TOKEN")
    login(token=HF_TOKEN)
    print("✅ Successfully logged in to Hugging Face using Kaggle Secret.")
except (ImportError, Exception):
    print("Kaggle secrets not found. Please enter your Hugging Face token.")
    try:
        token = getpass("Enter your Hugging Face token: ")
        login(token=token)
        print("✅ Successfully logged in to Hugging Face.")
    except Exception as e:
        print(f"❌ Failed to log in: {e}")

# Install required packages with fixed versions
!pip install --upgrade \
    "chromadb~=1.0.1" \
    "transformers>=4.41.0" \
    "torch==2.6.0" \
    "torchvision==0.21.0" \
    "torchaudio==2.6.0" --extra-index-url https://download.pytorch.org/whl/cu121 \
    "pdf2image" \
    "reportlab" \
    "accelerate>=0.31.0" \
    "bitsandbytes" \
    "sentence-transformers==3.0.0" \
    "colpali-engine>=0.3.10" \
    "pynvml"

# %%
# 2) Import libraries and setup
# -----------------------------
import os
import uuid
import logging
import gc
import json
import time
import warnings
import traceback
import threading
import math
import psutil
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor, AutoModelForImageTextToText
from sentence_transformers import SentenceTransformer
import chromadb
from pdf2image import convert_from_path
from PIL import Image, ImageDraw
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter

# GPU monitoring utilities
try:
    import pynvml
    pynvml.nvmlInit()
    NVML_AVAILABLE = True
except ImportError:
    NVML_AVAILABLE = False
    logging.warning("pynvml not available. GPU monitoring will be limited.")

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# %%
# 3) GPU Configuration and Monitoring
# -----------------------------------
class GPUConfig:
    """Configuration for GPU optimization."""
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.use_fp16 = torch.cuda.is_available()
        self.max_batch_size = 4 if torch.cuda.is_available() else 1
        self.enable_memory_efficient_attention = False

def get_gpu_utilization() -> Tuple[int, int]:
    """Get current GPU utilization and memory usage."""
    if not torch.cuda.is_available():
        return 0, 0
    
    try:
        if NVML_AVAILABLE:
            handle = pynvml.nvmlDeviceGetHandleByIndex(0)
            util = pynvml.nvmlDeviceGetUtilizationRates(handle)
            mem_info = pynvml.nvmlDeviceGetMemoryInfo(handle)
            return util.gpu, int(mem_info.used / mem_info.total * 100)
        else:
            return 0, 0
    except Exception:
        return 0, 0

def force_gpu_computation():
    """Force GPU computation to warm up and ensure utilization."""
    if torch.cuda.is_available():
        device = torch.device("cuda")
        for _ in range(3):
            a = torch.randn(1000, 1000, device=device, dtype=torch.float16)
            b = torch.randn(1000, 1000, device=device, dtype=torch.float16)
            c = torch.matmul(a, b)
            c = torch.relu(c)
            torch.cuda.synchronize()
            del a, b, c

# Initialize GPU configuration
gpu_config = GPUConfig()
device = gpu_config.device
logging.info(f"Using device: {device}")

# %%
# 4) Directory setup and constants
# --------------------------------
BASE_DIR = Path("/teamspace/studios/this_studio")
DATA_DIR = BASE_DIR / "data"
IMG_DIR = BASE_DIR / "storage" / "images"
DB_DIR = BASE_DIR / "storage" / "chroma_db"
OUTPUT_DIR = BASE_DIR / "output"

# Leishmania-specific keywords
LEISHMANIA_KEYWORDS = [
    'leishmaniasis', 'leishmania', 'kala-azar', 'visceral leishmaniasis',
    'cutaneous leishmaniasis', 'mucocutaneous leishmaniasis',
    'sandfly', 'phlebotomus', 'lutzomyia', 'amastigotes', 'promastigotes',
    'montenegro test', 'pentavalent antimony', 'amphotericin b',
    'miltefosine', 'chiclero', 'espundia', 'oriental sore',
    'leishmania major', 'leishmania donovani', 'leishmania infantum',
    'leishmania tropica', 'leishmania braziliensis', 'leishmania mexicana'
]

TOP_K = 5  # Number of documents to retrieve

for d in (DATA_DIR, IMG_DIR, DB_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# %%
# 5) Utility functions
# --------------------
def safe_filename(filepath: Path) -> str:
    """Create a safe filename for saving images."""
    safe_name = str(filepath.stem)
    problematic_chars = ['/', '\\', ':', '*', '?', '"', '<', '>', '|', ' ', '-', '(', ')', '[', ']']
    for char in problematic_chars:
        safe_name = safe_name.replace(char, '_')
    while '__' in safe_name:
        safe_name = safe_name.replace('__', '_')
    return safe_name[:100] if len(safe_name) > 100 else safe_name

def is_leishmania_related(text: str) -> bool:
    """Check if text contains Leishmania-related keywords."""
    text_lower = text.lower()
    return any(keyword in text_lower for keyword in LEISHMANIA_KEYWORDS)

def find_all_pdfs(directory: Path) -> List[Path]:
    """Recursively find all PDF files in directory and subdirectories."""
    pdf_files = []
    
    def scan_directory(path: Path):
        try:
            for item in path.iterdir():
                if item.is_file() and item.suffix.lower() == '.pdf':
                    pdf_files.append(item)
                elif item.is_dir():
                    scan_directory(item)
        except PermissionError:
            logging.warning(f"Permission denied accessing: {path}")
        except Exception as e:
            logging.warning(f"Error scanning directory {path}: {e}")
    
    scan_directory(directory)
    return pdf_files

# %%
# 6) PDF processing functions
# ---------------------------
def process_single_pdf_optimized(pdf_path: Path, img_dir: Path, max_workers: int = 4) -> Tuple[List[str], bool]:
    """Process a single PDF file with optimized settings."""
    page_images = []
    success = True
    
    try:
        logging.info(f"Processing PDF: {pdf_path.name}")
        
        # Optimized DPI settings
        dpi_settings = [150, 100, 200, 72]
        pages = None
        
        for dpi in dpi_settings:
            try:
                pages = convert_from_path(
                    str(pdf_path), 
                    dpi=dpi,
                    thread_count=max_workers,
                    fmt='PNG',
                    strict=False
                )
                logging.info(f"Successfully converted {pdf_path.name} with DPI={dpi}")
                break
            except Exception as e:
                logging.warning(f"Failed to convert {pdf_path.name} with DPI={dpi}: {e}")
                continue
        
        if pages is None:
            logging.error(f"Failed to convert {pdf_path.name} with all DPI settings")
            return [], False
        
        # Save pages with optimization
        safe_name = safe_filename(pdf_path)
        
        for i, page in enumerate(pages):
            try:
                img_filename = f"{safe_name}_page{i+1:03d}.png"
                img_path = img_dir / img_filename
                
                if page.mode != 'RGB':
                    page = page.convert('RGB')
                
                # Resize if too large
                if max(page.size) > 2048:
                    page.thumbnail((2048, 2048), Image.Resampling.LANCZOS)
                
                page.save(img_path, "PNG", optimize=True, compress_level=6)
                page_images.append(str(img_path))
                
            except Exception as e:
                logging.error(f"Failed to save page {i+1} of {pdf_path.name}: {e}")
                success = False
        
        # Sort to maintain page order
        page_images.sort(key=lambda x: int(x.split('_page')[1].split('.')[0]))
        logging.info(f"Successfully processed {len(page_images)} pages from {pdf_path.name}")
        
    except Exception as e:
        logging.error(f"Critical error processing {pdf_path.name}: {e}")
        success = False
    
    return page_images, success

def process_all_pdfs_optimized():
    """Find and process all PDFs with optimized settings."""
    all_pdfs = find_all_pdfs(DATA_DIR)
    
    if not all_pdfs:
        logging.warning("No PDFs found. Creating demo medical report with Leishmania content.")
        dummy_pdf_path = DATA_DIR / "leishmania_medical_report.pdf"
        if not dummy_pdf_path.exists():
            c = canvas.Canvas(str(dummy_pdf_path), pagesize=letter)
            width, height = letter
            c.drawString(72, height - 72, "Patient Report: Leishmaniasis Case Study")
            c.drawString(72, height - 100, "Diagnosis: Cutaneous Leishmaniasis")
            c.drawString(72, height - 130, "Causative agent: Leishmania major")
            c.drawString(72, height - 160, "Treatment: Pentavalent antimony, Amphotericin B")
            c.showPage()
            c.drawString(72, height - 72, "Laboratory Findings")
            c.drawString(72, height - 100, "Montenegro test: Positive")
            c.drawString(72, height - 130, "PCR for Leishmania: Positive")
            c.drawString(72, height - 160, "Microscopy: Amastigotes identified")
            c.showPage()
            c.save()
            logging.info(f"Created demo PDF: {dummy_pdf_path}")
        all_pdfs = [dummy_pdf_path]
    else:
        logging.info(f"Found {len(all_pdfs)} PDF(s) in data folder")
    
    return all_pdfs

# %%
# 7) GPU-Optimized ColPali for retrieval
# --------------------------------------
from colpali_engine.models import ColPali
from colpali_engine.utils.processing_utils import BaseProcessor

COLPALI_BASE_ID = "google/paligemma-3b-pt-448"
COLPALI_ADAPTER_ID = "vidore/colpali-v1.2"

class GPUOptimizedColPali:
    """GPU-optimized ColPali implementation."""
    
    def __init__(self, base_model_id: str, adapter_id: str, gpu_config: GPUConfig):
        self.gpu_config = gpu_config
        self.device = gpu_config.device
        self.base_model_id = base_model_id
        self.adapter_id = adapter_id
        
        force_gpu_computation()
        self._load_model()
    
    def _load_model(self):
        """Load ColPali model with proper error handling."""
        try:
            logging.info(f"Loading ColPali model...")
            
            # Load ColPali model
            self.model = ColPali.from_pretrained(
                self.adapter_id,
                torch_dtype=torch.bfloat16,
                device_map={"": self.device}
            ).eval()
            
            # Load processor
            self.processor = BaseProcessor.from_pretrained(self.adapter_id)
            
            logging.info(f"✅ ColPali loaded successfully on {self.device}")
            
        except Exception as e:
            logging.error(f"Failed to load ColPali: {e}")
            self._create_fallback_model()
    
    def _create_fallback_model(self):
        """Create a simple GPU-based embedding fallback."""
        logging.info("Creating GPU embedding fallback...")
        
        class GPUEmbeddingFallback(torch.nn.Module):
            def __init__(self, embedding_dim=128):
                super().__init__()
                self.vision_encoder = torch.nn.Sequential(
                    torch.nn.Linear(224*224*3, 1024),
                    torch.nn.ReLU(),
                    torch.nn.Linear(1024, embedding_dim)
                )
                self.text_encoder = torch.nn.Sequential(
                    torch.nn.Embedding(50000, 512),
                    torch.nn.Linear(512, embedding_dim)
                )
        
        self.model = GPUEmbeddingFallback().to(self.device)
        
        class SimpleTokenizer:
            def __call__(self, texts, images=None, **kwargs):
                if isinstance(texts, str):
                    texts = [texts]
                
                tokenized = []
                for text in texts:
                    words = text.lower().split()
                    ids = [hash(word) % 50000 for word in words[:50]]
                    tokenized.append(ids)
                
                max_len = max(len(t) for t in tokenized) if tokenized else 1
                for t in tokenized:
                    t.extend([0] * (max_len - len(t)))
                
                return {
                    'input_ids': torch.tensor(tokenized, device=self.device),
                    'attention_mask': torch.ones(len(tokenized), max_len, device=self.device)
                }
        
        self.processor = SimpleTokenizer()
    
    def embed_pages_gpu(self, image_paths: List[str], batch_size: Optional[int] = None) -> Tuple[np.ndarray, List[str]]:
        """GPU-optimized batch embedding."""
        if not image_paths:
            return np.array([]), []
        
        batch_size = batch_size or self.gpu_config.max_batch_size
        
        # Preprocess images
        valid_images, valid_paths = [], []
        for path in image_paths:
            try:
                if os.path.exists(path):
                    img = Image.open(path).convert("RGB")
                    if max(img.size) > 1024:
                        img.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
                    valid_images.append(img)
                    valid_paths.append(path)
            except Exception as e:
                logging.warning(f"Failed to load image {path}: {e}")
        
        if not valid_images:
            return np.array([]), []
        
        all_embeddings = []
        
        # Process in batches
        for i in range(0, len(valid_images), batch_size):
            batch_images = valid_images[i:i + batch_size]
            
            try:
                if hasattr(self.model, 'vision_encoder'):
                    # Using fallback model
                    embeddings = []
                    for img in batch_images:
                        img_array = np.array(img).astype(np.float32) / 255.0
                        img_tensor = torch.tensor(
                            img_array.flatten(),
                            device=self.device,
                            dtype=torch.float32
                        )
                        with torch.cuda.amp.autocast():
                            emb = self.model.vision_encoder(img_tensor)
                            emb = F.normalize(emb, dim=-1)
                        embeddings.append(emb)
                    result = torch.stack(embeddings)
                else:
                    # Using actual ColPali model
                    batch_texts = ["Document image" for _ in batch_images]
                    
                    with torch.cuda.amp.autocast(enabled=self.gpu_config.use_fp16):
                        batch_inputs = self.processor(
                            text=batch_texts,
                            images=batch_images,
                            return_tensors="pt"
                        )
                        
                        for key in batch_inputs:
                            if isinstance(batch_inputs[key], torch.Tensor):
                                batch_inputs[key] = batch_inputs[key].to(self.device)
                        
                        with torch.no_grad():
                            outputs = self.model(**batch_inputs)
                            
                            if hasattr(outputs, 'last_hidden_state'):
                                embeddings = outputs.last_hidden_state
                            else:
                                embeddings = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
                            
                            embeddings = embeddings.mean(dim=1)
                            result = F.normalize(embeddings, p=2, dim=1)
                
                final_embeddings = result.cpu().float().numpy()
                all_embeddings.append(final_embeddings)
                
            except Exception as e:
                logging.error(f"Error processing batch {i//batch_size}: {e}")
                fallback_dim = 128
                zero_embeds = np.zeros((len(batch_images), fallback_dim), dtype=np.float32)
                all_embeddings.append(zero_embeds)
            
            # Memory management
            if i % (batch_size * 2) == 0:
                torch.cuda.empty_cache()
        
        if all_embeddings:
            final_embeddings = np.concatenate(all_embeddings, axis=0)
            logging.info(f"Generated {final_embeddings.shape[0]} embeddings")
        else:
            final_embeddings = np.array([])
        
        return final_embeddings, valid_paths

    def embed_queries_gpu(self, texts: List[str]) -> np.ndarray:
        """GPU-optimized query embedding."""
        if isinstance(texts, str):
            texts = [texts]
        
        force_gpu_computation()
        
        try:
            if hasattr(self.model, 'text_encoder'):
                # Using fallback model
                embeddings = []
                for query in texts:
                    words = query.lower().split()
                    ids = [hash(word) % 50000 for word in words[:50]]
                    ids = ids + [0] * (50 - len(ids))
                    
                    query_ids = torch.tensor([ids], device=self.device)
                    
                    with torch.cuda.amp.autocast():
                        emb = self.model.text_encoder(query_ids)
                        emb = F.normalize(emb, dim=-1)
                        embeddings.append(emb.squeeze(0))
                
                result = torch.stack(embeddings)
            else:
                # Using actual ColPali model
                with torch.cuda.amp.autocast(enabled=self.gpu_config.use_fp16):
                    inputs = self.processor(text=texts, return_tensors="pt")
                    
                    for key in inputs:
                        if isinstance(inputs[key], torch.Tensor):
                            inputs[key] = inputs[key].to(self.device)
                    
                    with torch.no_grad():
                        outputs = self.model(**inputs)
                        
                        if hasattr(outputs, 'last_hidden_state'):
                            embeddings = outputs.last_hidden_state
                        else:
                            embeddings = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
                        
                        embeddings = embeddings.mean(dim=1)
                        result = F.normalize(embeddings, p=2, dim=1)
            
            return result.cpu().float().numpy()
                    
        except Exception as e:
            logging.error(f"Error in GPU query embedding: {e}")
            return np.zeros((len(texts), 128), dtype=np.float32)

# Initialize ColPali
colpali = GPUOptimizedColPali(COLPALI_BASE_ID, COLPALI_ADAPTER_ID, gpu_config)

# %%
# 8) Smart indexing with Leishmania filtering
# -------------------------------------------
client = chromadb.PersistentClient(path=str(DB_DIR))

# Create collections
leishmania_col = client.get_or_create_collection("leishmania_pages")
general_col = client.get_or_create_collection("general_pages")

def smart_content_filtering(image_path: str) -> Dict[str, Any]:
    """Analyze image content and determine if it's Leishmania-related."""
    filename = Path(image_path).stem.lower()
    is_leishmania = is_leishmania_related(filename)
    
    return {
        "is_leishmania": is_leishmania,
        "confidence": 0.8 if is_leishmania else 0.2,
        "filename": filename
    }

def build_smart_index():
    """Build or update the index with smart content filtering."""
    logging.info("Checking for new documents to index...")

    # Get all PDFs
    all_pdfs_on_disk = process_all_pdfs_optimized()
    if not all_pdfs_on_disk:
        logging.warning("No PDF documents found.")
        return

    # Get already indexed PDFs
    indexed_pdf_stems = set()
    try:
        if leishmania_col.count() > 0:
            leish_metas = leishmania_col.get(include=["metadatas"])
            for meta in leish_metas.get('metadatas', []):
                if 'pdf' in meta:
                    indexed_pdf_stems.add(meta['pdf'])

        if general_col.count() > 0:
            gen_metas = general_col.get(include=["metadatas"])
            for meta in gen_metas.get('metadatas', []):
                if 'pdf' in meta:
                    indexed_pdf_stems.add(meta['pdf'])
    except Exception as e:
        logging.error(f"Could not retrieve metadata from ChromaDB: {e}")
        indexed_pdf_stems = set()

    # Identify new PDFs
    pdfs_to_process = []
    for pdf_path in all_pdfs_on_disk:
        safe_stem = safe_filename(pdf_path)
        if safe_stem not in indexed_pdf_stems:
            pdfs_to_process.append(pdf_path)

    if not pdfs_to_process:
        logging.info("✅ Index is up-to-date.")
        return

    logging.info(f"Found {len(pdfs_to_process)} new document(s) to index")
    
    # Process new PDFs
    all_page_images = []
    for pdf_path in pdfs_to_process:
        page_images, success = process_single_pdf_optimized(pdf_path, IMG_DIR)
        if success and page_images:
            all_page_images.extend(page_images)
    
    if not all_page_images:
        logging.warning("No new pages extracted.")
        return
    
    logging.info(f"Processing {len(all_page_images)} new pages...")
    
    # Smart filtering
    leishmania_images = []
    general_images = []
    
    for img_path in all_page_images:
        content_info = smart_content_filtering(img_path)
        if content_info["is_leishmania"]:
            leishmania_images.append(img_path)
        else:
            general_images.append(img_path)
    
    logging.info(f"Leishmania-related: {len(leishmania_images)}, General: {len(general_images)}")
    
    # Index Leishmania content
    if leishmania_images:
        embeddings, valid_paths = colpali.embed_pages_gpu(leishmania_images)
        
        if len(embeddings) > 0:
            page_ids = [str(uuid.uuid4()) for _ in valid_paths]
            metadatas = []
            
            for img_path in valid_paths:
                img_path_obj = Path(img_path)
                pdf_name = img_path_obj.stem.split('_page')[0]
                page_num = img_path_obj.stem.split('_page')[-1] if '_page' in img_path_obj.stem else "1"
                
                metadatas.append({
                    "pdf": pdf_name,
                    "image_path": img_path,
                    "page_number": page_num,
                    "content_type": "leishmania",
                    "priority": "high"
                })
            
            leishmania_col.add(
                ids=page_ids,
                embeddings=embeddings.tolist(),
                documents=valid_paths,
                metadatas=metadatas
            )
            logging.info(f"✅ Indexed {len(valid_paths)} Leishmania pages")
    
    # Index general content (sample)
    if general_images:
        sample_size = min(len(general_images), 200)
        sampled_general = general_images[:sample_size]
        
        embeddings, valid_paths = colpali.embed_pages_gpu(sampled_general)
        
        if len(embeddings) > 0:
            page_ids = [str(uuid.uuid4()) for _ in valid_paths]
            metadatas = []
            
            for img_path in valid_paths:
                img_path_obj = Path(img_path)
                pdf_name = img_path_obj.stem.split('_page')[0]
                page_num = img_path_obj.stem.split('_page')[-1] if '_page' in img_path_obj.stem else "1"
                
                metadatas.append({
                    "pdf": pdf_name,
                    "image_path": img_path,
                    "page_number": page_num,
                    "content_type": "general",
                    "priority": "medium"
                })
            
            general_col.add(
                ids=page_ids,
                embeddings=embeddings.tolist(),
                documents=valid_paths,
                metadatas=metadatas
            )
            logging.info(f"✅ Indexed {len(valid_paths)} general pages")
    
    # Cleanup
    torch.cuda.empty_cache()
    gc.collect()

# Build index
build_smart_index()

# %%
# 9) GPU-Optimized MedGemma for answer generation
# -----------------------------------------------
MED_ID = "google/medgemma-4b-it"

class GPUOptimizedMedGemma:
    """GPU-optimized MedGemma implementation."""
    
    def __init__(self, model_id: str, gpu_config: GPUConfig):
        self.gpu_config = gpu_config
        self.device = gpu_config.device
        self.model_id = model_id
        
        try:
            logging.info(f"Loading MedGemma: {model_id}")
            
            self.processor = AutoProcessor.from_pretrained(
                model_id,
                trust_remote_code=True
            )
            
            self.model = AutoModelForImageTextToText.from_pretrained(
                model_id,
                trust_remote_code=True,
                torch_dtype=torch.float16 if gpu_config.use_fp16 else torch.float32,
                device_map="auto",
                low_cpu_mem_usage=True
            )
            
            self.model = self.model.to(self.device)
            self.model.eval()
            
            if self.processor.tokenizer.pad_token is None:
                self.processor.tokenizer.pad_token = self.processor.tokenizer.eos_token
            
            logging.info(f"✅ MedGemma loaded successfully on {self.device}")
            
        except Exception as e:
            logging.error(f"Failed to load MedGemma: {e}")
            raise

    def generate_answer_gpu(self, images: List[Image.Image], prompt: str) -> str:
        """GPU-optimized answer generation."""
        try:
            # Preprocess images
            processed_images = []
            max_images = 3
            
            for img in images[:max_images]:
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                if max(img.size) > 896:
                    img.thumbnail((896, 896), Image.Resampling.LANCZOS)
                processed_images.append(img)
            
            # Force GPU computation
            force_gpu_computation()
            
            if not processed_images:
                inputs = self.processor(
                    text=prompt,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=1024
                )
            else:
                inputs = self.processor(
                    text=prompt,
                    images=processed_images,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=1024
                )
            
            # Move to GPU
            for key in inputs:
                if isinstance(inputs[key], torch.Tensor):
                    inputs[key] = inputs[key].to(self.device, non_blocking=True)
            
            # Generation
            with torch.no_grad():
                with torch.cuda.amp.autocast(enabled=self.gpu_config.use_fp16):
                    outputs = self.model.generate(
                        **inputs,
                        max_new_tokens=512,
                        do_sample=True,
                        temperature=0.7,
                        top_p=0.9,
                        pad_token_id=self.processor.tokenizer.pad_token_id,
                        eos_token_id=self.processor.tokenizer.eos_token_id
                    )
            
            # Decode response
            input_length = inputs['input_ids'].shape[1]
            generated_tokens = outputs[0][input_length:]
            response = self.processor.tokenizer.decode(
                generated_tokens, 
                skip_special_tokens=True
            ).strip()
            
            return response or "I couldn't generate a proper response. Please try rephrasing."
            
        except Exception as e:
            logging.error(f"Error in generation: {e}")
            torch.cuda.empty_cache()
            return f"Error generating response: {str(e)}"

# Initialize MedGemma
medgemma = GPUOptimizedMedGemma(MED_ID, gpu_config)

# %%
# 10) Query processing functions
# -----------------------------
def process_multimodal_query(text_query: str, image_paths: List[str]) -> np.ndarray:
    """Process multimodal query with GPU computation."""
    try:
        logging.info(f"Processing multimodal query: {len(image_paths)} images")
        
        text_embedding = colpali.embed_queries_gpu([text_query])
        
        if image_paths:
            valid_image_paths = [p for p in image_paths if os.path.exists(p)]
            
            if valid_image_paths:
                image_embeddings, _ = colpali.embed_pages_gpu(valid_image_paths)
                
                if len(image_embeddings) > 0:
                    text_weight = 0.7
                    image_weight = 0.3
                    
                    avg_image_embedding = np.mean(image_embeddings, axis=0, keepdims=True)
                    
                    if text_embedding.shape == avg_image_embedding.shape:
                        combined_embedding = (text_weight * text_embedding + 
                                           image_weight * avg_image_embedding)
                    else:
                        combined_embedding = text_embedding
                    
                    return combined_embedding
        
        return text_embedding
        
    except Exception as e:
        logging.error(f"Error in multimodal query: {e}")
        return colpali.embed_queries_gpu([text_query])

def smart_query_system(query: str, query_images: Optional[List[str]] = None, 
                      top_k: int = TOP_K, prioritize_leishmania: bool = True, 
                      return_images: bool = True) -> Dict[str, Any]:
    """Enhanced smart query system with multimodal support."""
    start_time = time.time()
    query_images = query_images or []
    
    is_leishmania_query = is_leishmania_related(query)
    
    try:
        logging.info(f"Processing query: '{query}' (Leishmania: {is_leishmania_query})")
        
        # Process multimodal query
        if query_images:
            query_embedding = process_multimodal_query(query, query_images)
        else:
            query_embedding = colpali.embed_queries_gpu([query])
        
        if query_embedding.size == 0:
            return {"text": "Failed to embed query.", "images": [], "metadata": {}}
        
        # Retrieve documents
        retrieved_docs = []
        retrieved_metas = []
        
        if is_leishmania_query and prioritize_leishmania:
            # Prioritize Leishmania content
            if leishmania_col.count() > 0:
                leish_results = leishmania_col.query(
                    query_embeddings=query_embedding.tolist(),
                    n_results=min(top_k, leishmania_col.count())
                )
                if leish_results["documents"][0]:
                    retrieved_docs.extend(leish_results["documents"][0])
                    retrieved_metas.extend(leish_results["metadatas"][0])
            
            # Add general content if needed
            if len(retrieved_docs) < top_k and general_col.count() > 0:
                remaining = top_k - len(retrieved_docs)
                gen_results = general_col.query(
                    query_embeddings=query_embedding.tolist(),
                    n_results=min(remaining, general_col.count())
                )
                if gen_results["documents"][0]:
                    retrieved_docs.extend(gen_results["documents"][0])
                    retrieved_metas.extend(gen_results["metadatas"][0])
        else:
            # Search both collections
            total_results = []
            
            if leishmania_col.count() > 0:
                leish_results = leishmania_col.query(
                    query_embeddings=query_embedding.tolist(), 
                    n_results=min(top_k, leishmania_col.count())
                )
                if leish_results["documents"][0]:
                    for i, doc in enumerate(leish_results["documents"][0]):
                        total_results.append((doc, leish_results["metadatas"][0][i], leish_results["distances"][0][i]))
            
            if general_col.count() > 0:
                gen_results = general_col.query(
                    query_embeddings=query_embedding.tolist(), 
                    n_results=min(top_k, general_col.count())
                )
                if gen_results["documents"][0]:
                    for i, doc in enumerate(gen_results["documents"][0]):
                        total_results.append((doc, gen_results["metadatas"][0][i], gen_results["distances"][0][i]))
            
            total_results.sort(key=lambda x: x[2])
            total_results = total_results[:top_k]
            retrieved_docs = [result[0] for result in total_results]
            retrieved_metas = [result[1] for result in total_results]
        
        if not retrieved_docs:
            return {"text": "No relevant documents found.", "images": [], "metadata": {}}
        
        # Load and validate images
        valid_images = []
        valid_image_paths = []
        valid_metas = []
        
        for doc_path, meta in zip(retrieved_docs, retrieved_metas):
            try:
                if os.path.exists(doc_path):
                    img = Image.open(doc_path).convert("RGB")
                    valid_images.append(img)
                    valid_image_paths.append(doc_path)
                    valid_metas.append(meta)
            except Exception as e:
                logging.warning(f"Failed to load image {doc_path}: {e}")
        
        if not valid_images:
            return {"text": "Retrieved documents could not be loaded.", "images": [], "metadata": {}}
        
        # Prepare context for generation
        all_context_images = valid_images.copy()
        if query_images:
            query_imgs = [Image.open(p).convert("RGB") for p in query_images if os.path.exists(p)]
            all_context_images = query_imgs + all_context_images
        
        # Generate answer
        leishmania_count = sum(1 for meta in valid_metas if meta.get("content_type") == "leishmania")
        context_info = f"(based on {len(valid_images)} pages, {leishmania_count} Leishmania-specific)"
        if query_images:
            context_info += f" and {len(query_images)} provided image(s)"

        prompt = f"""<start_of_turn>user
Answer the question based on the provided medical document pages.
Context: {context_info}
Question: {query}<end_of_turn>
<start_of_turn>model
"""
        
        logging.info("Generating answer with MedGemma...")
        answer = medgemma.generate_answer_gpu(all_context_images, prompt)
        
        # Prepare response
        response_images = []
        if return_images:
            for i, img_path in enumerate(valid_image_paths[:3]):
                response_images.append({
                    "path": img_path,
                    "metadata": valid_metas[i],
                    "relevance_rank": i + 1
                })
        
        # Format final response
        processing_time = time.time() - start_time
        source_info = f"\n\n📄 Sources: {len(valid_images)} pages ({leishmania_count} Leishmania-specific)"
        if query_images:
            source_info += f"\n🖼️ Query images: {len(query_images)} analyzed"
        source_info += f"\n⚡ Processing time: {processing_time:.2f}s"
        
        return {
            "text": answer + source_info,
            "images": response_images,
            "metadata": {
                "query": query,
                "query_images": query_images,
                "leishmania_related": is_leishmania_query,
                "processing_time": processing_time,
                "sources_count": len(valid_images),
                "leishmania_sources": leishmania_count
            }
        }
        
    except Exception as e:
        logging.error(f"Error in smart_query_system: {e}")
        torch.cuda.empty_cache()
        return {"text": f"Error: {str(e)}", "images": [], "metadata": {"error": str(e)}}

# %%
# 11) Response handling functions
# ------------------------------
def display_multimodal_response(result: Dict[str, Any]):
    """Display a multimodal response with proper formatting."""
    print("\n" + "="*70)
    print("           MULTIMODAL RESPONSE")
    print("="*70)
    
    # Display text response
    print(f"💬 Text Response:")
    print(f"{result.get('text', 'No text response available.')}")
    
    # Display images
    if result.get('images'):
        print(f"\n🖼️ Visual Evidence ({len(result['images'])} images):")
        print("-" * 50)
        for i, img_info in enumerate(result['images'], 1):
            print(f"  📸 Image {i}: {os.path.basename(img_info.get('path', ''))}")
            meta = img_info.get('metadata', {})
            print(f"     Source: {meta.get('pdf', 'unknown')} | Page: {meta.get('page_number', 'unknown')}")
    
    # Display metadata
    metadata = result.get('metadata', {})
    if metadata:
        print(f"\n📊 Processing Info:")
        print(f"   - Processing time: {metadata.get('processing_time', 0):.2f}s")
        print(f"   - Leishmania query: {metadata.get('leishmania_related', False)}")
        print(f"   - Sources: {metadata.get('sources_count', 0)} pages")
    print("="*70)

def save_multimodal_response(result: Dict[str, Any], output_dir: Path = OUTPUT_DIR):
    """Save a multimodal response to files."""
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = int(time.time())
    
    try:
        response_file = output_dir / f"response_{timestamp}.json"
        with open(response_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=4)
        
        if result.get('images'):
            img_dir = output_dir / f"images_{timestamp}"
            img_dir.mkdir(exist_ok=True)
            for img_info in result['images']:
                src_path = Path(img_info['path'])
                if src_path.exists():
                    import shutil
                    dst_path = img_dir / src_path.name
                    shutil.copy(src_path, dst_path)
        
        logging.info(f"✅ Response saved to {response_file}")
        return str(response_file)
        
    except Exception as e:
        logging.error(f"Error saving response: {e}")
        return None

# %%
# 12) Testing and utility functions
# ---------------------------------
def run_tests():
    """Run comprehensive tests."""
    print("\n" + "="*60)
    print("     MULTIMODAL RAG SYSTEM - TEST SUITE")
    print("="*60 + "\n")
    
    # Create dummy test image
    dummy_image_path = DATA_DIR / "test_lesion.png"
    if not dummy_image_path.exists():
        try:
            img = Image.new('RGB', (200, 200), color='pink')
            draw = ImageDraw.Draw(img)
            draw.ellipse((50, 50, 150, 150), fill='red', outline='darkred')
            draw.text((10, 10), "Sample Lesion", fill="black")
            img.save(dummy_image_path)
            print(f"🖼️ Created test image: {dummy_image_path}")
        except:
            dummy_image_path = None
    
    test_cases = [
        {
            "description": "Leishmania Text Query",
            "query": "What are the clinical features of cutaneous leishmaniasis?",
            "query_images": None
        },
        {
            "description": "General Medical Query",
            "query": "What are the symptoms of malaria?",
            "query_images": None
        },
        {
            "description": "Multimodal Query",
            "query": "Analyze this skin lesion.",
            "query_images": [str(dummy_image_path)] if dummy_image_path else None
        }
    ]
    
    for i, case in enumerate(test_cases, 1):
        print(f"\n--- Test {i}: {case['description']} ---")
        
        if case.get("query_images") is None and "Multimodal" in case["description"]:
            print("⚠️ Skipping: Test image not available")
            continue
        
        try:
            result = smart_query_system(
                query=case['query'],
                query_images=case['query_images'],
                top_k=2
            )
            display_multimodal_response(result)
            
        except Exception as e:
            print(f"❌ Test failed: {e}")
        
        print("-" * 50)

def interactive_system():
    """Interactive query system."""
    print("\n" + "="*70)
    print("     INTERACTIVE LEISHMANIA RAG SYSTEM")
    print("="*70)
    print("🦠 Specialized for Leishmania research")
    print("🚀 GPU-accelerated processing")
    print("💡 Type 'help' for commands, 'quit' to exit")
    print("="*70 + "\n")
    
    while True:
        try:
            query = input("🔍 Your question: ").strip()
            
            if not query:
                continue
            
            if query.lower() in ['quit', 'exit', 'q']:
                print("👋 Thank you!")
                break
                
            if query.lower() == 'help':
                print("\n📚 Commands:")
                print("  - Ask any question")
                print("  - 'stats': Show database statistics")
                print("  - 'test': Run test suite")
                print("  - 'quit': Exit")
                continue
            
            if query.lower() == 'stats':
                print(f"\n📊 Statistics:")
                print(f"  - Leishmania pages: {leishmania_col.count()}")
                print(f"  - General pages: {general_col.count()}")
                continue
            
            if query.lower() == 'test':
                run_tests()
                continue
            
            # Handle multimodal input
            image_input = input("🖼️ Image paths (optional, comma-separated): ").strip()
            query_images = []
            if image_input:
                for path in image_input.split(','):
                    p = Path(path.strip())
                    if p.exists():
                        query_images.append(str(p))
                        print(f"  ✅ Added: {p.name}")
                    else:
                        print(f"  ❌ Not found: {p}")
            
            print(f"\n⏳ Processing...")
            
            is_leish_query = is_leishmania_related(query)
            if is_leish_query:
                print("🦠 Leishmania query detected")
            
            result = smart_query_system(
                query, 
                query_images=query_images, 
                prioritize_leishmania=is_leish_query
            )
            
            display_multimodal_response(result)
            
            save_q = input("💾 Save response? (y/n): ").strip().lower()
            if save_q == 'y':
                save_multimodal_response(result)

        except KeyboardInterrupt:
            print("\n👋 Exiting...")
            break
        except Exception as e:
            print(f"❌ Error: {e}")
            torch.cuda.empty_cache()

# %%
# 13) System summary and ready message
# -----------------------------------
def show_system_summary():
    """Display system summary."""
    print("\n" + "="*60)
    print("           SYSTEM SUMMARY")
    print("="*60)
    
    print(f"📊 Database: {leishmania_col.count()} Leishmania, {general_col.count()} general pages")
    
    if device.type == 'cuda':
        gpu_name = torch.cuda.get_device_name(0)
        mem_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"🚀 GPU: {gpu_name} ({mem_total:.1f} GB)")
    else:
        print(f"❌ GPU: Not available")
    
    print(f"🤖 Models: ColPali (retrieval), MedGemma (generation)")
    print(f"🖼️ Multimodal: ✅ Text+Image input/output")
    print("="*60)

show_system_summary()

print("\n🎉 GPU-Optimized Multimodal Leishmania RAG System Ready!")
print("📚 Documents processed and indexed")
print("🦠 Leishmania content prioritized")
print("\nTo start:")
print("  1. interactive_system()  <- Interactive mode")
print("  2. run_tests()           <- Test functionality")
print("  3. smart_query_system(query='Your question')")